# Notebook 1 — Segment images with Cellpose 4.0.6 and save masks

## Installation 

### for Mac
Latest stable release is Cellpose 4.0.6, [available via conda-forge](https://anaconda.org/conda-forge/cellpose)

```bash
conda env create -f ./envs/cellpose.yml
conda activate cellpose
```

### for colab

`%pip install "cellpose==4.0.6" "torch" "torchvision" "torchaudio" "scikit-image>=0.22.0" "tqdm>=4.66.0" "pandas>=2.2.0"`

In [1]:
# Cell 1 - Imports and Apple Silicon device detection
from pathlib import Path
from typing import List
import os
import numpy as np
from skimage import io
from tqdm import tqdm
import torch
from cellpose import models

def has_mps() -> bool:
    return hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

device = torch.device("mps") if has_mps() else torch.device("cpu")
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
print("torch:", torch.__version__)
print("device:", device)




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	darwin 
python version: 	3.10.18 
torch version:  	2.7.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


torch: 2.7.1
device: mps


In [2]:
# Cell 2 — Project paths and parameters
project_root = Path("/Users/ashi/github/cm4ai_codefest2025")

# Inputs: expect data/red, data/yellow, data/blue, data/green
img_root = project_root / "data"
channels: List[str] = ["red", "yellow", "blue", "green"]

# Outputs: masks will go to analysis/cellpose_results2/<channel>/png/*_masks.png
masks_root = project_root / "analysis" / "cellpose_results2"

print("- project root is:" , project_root)
print("- image root is:" , img_root)
print("- channels are:", channels)
print("- masks root is:" , masks_root)

# Image extensions to search
image_exts = [".tif", ".tiff", ".png", ".jpg", ".jpeg"]

# Segmentation parameters — IMPORTANT: diameter=None (not 0)
diameter = None
flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 64
batch_size = 4
debug_max_images = None  # set small int to smoke test



- project root is: /Users/ashi/github/cm4ai_codefest2025
- image root is: /Users/ashi/github/cm4ai_codefest2025/data
- channels are: ['red', 'yellow', 'blue', 'green']
- masks root is: /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2


In [3]:
# Cell 3 — Utilities
def discover_images(folder: Path, exts) -> list[Path]:
    files = []
    for ext in exts:
        files.extend(sorted(folder.glob(f"*{ext}")))
    # de-duplicate by stem
    seen, uniq = set(), []
    for p in files:
        if p.stem not in seen:
            uniq.append(p); seen.add(p.stem)
    return uniq

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]



1. In CellPose 4.x Which model should you use?

It depends on what you are segmenting:

Your data type	Recommended built-in model (pretrained_model)	Notes
Whole cell (cytoplasm)	"cyto2"	Most commonly used, trained on diverse cell images, handles cytoplasm + background.
Nuclei only	"nuclei"	If your data is DAPI or nuclear stain only.
Cyto3 (Cellpose 3.x)	"cyto3"	Newer version with image restoration; can give better masks on noisy / low-contrast images.
Cellpose-SAM (new, 2025)	"cpsam"	Very general, works on many modalities. Trained with SAM integration; sometimes over-segments.
Custom trained	Path to your own .npy model file	You can fine-tune and then pass the path.

For your use case (multiplex immunofluorescence of B cells / Tregs), I’d recommend starting with:
	•	"cyto2" if you want whole-cell segmentation (membrane+cytoplasm).
	•	"nuclei" if you only want nuclear regions.

If segmentation quality is poor, try "cyto3" or "cpsam" next.

⸻

2. Where are models saved?

When you call:

model = models.CellposeModel(pretrained_model="cyto2", device=device, gpu=False)

Cellpose will:
	1.	Check in your local cache directory (~/.cellpose/models/ on macOS).
	2.	If the weights aren’t there yet, it downloads from the Cellpose model zoo (hosted on GitHub/HuggingFace).
	3.	It saves them into ~/.cellpose/models/ for reuse.

So after the first run, you’ll find files like:

/Users/ashi/.cellpose/models/cyto2.npy
/Users/ashi/.cellpose/models/nuclei.npy
/Users/ashi/.cellpose/models/cyto3.npy
/Users/ashi/.cellpose/models/cpsam.npy


⸻

3. How to check which models are available locally

import os
from pathlib import Path

model_dir = Path.home() / ".cellpose" / "models"
print("Local Cellpose models:", os.listdir(model_dir))


⸻

4. TL;DR for you
	•	Start with "cyto2".
	•	Models download automatically and live under ~/.cellpose/models/.
	•	You can swap to "nuclei", "cyto3", or "cpsam" by just changing the string.


In [3]:
from pathlib import Path, os
model_dir = Path.home() / ".cellpose" / "models"
print("Local Cellpose models:", os.listdir(model_dir))


Local Cellpose models: ['cpsam']


In [ ]:
# Cell 4 — Load Cellpose model (4.x API)


# Model choice: "cyto2" (whole cells), "nuclei", "cyto3", or "cpsam"
pretrained_model = "cyto2"


# Keep gpu=False; MPS goes via torch 'device'
model = models.CellposeModel(
    gpu=False,
    pretrained_model=pretrained_model,
    device=device
)
print("Loaded model:", model.pretrained_model)


pretrained model /Users/ashi/.cellpose/models/cpsam not found, using default model


Loaded model: /Users/ashi/.cellpose/models/cpsam


In [5]:
# Cell 5 — Run segmentation per channel and write *_masks.png
# Assumes single-channel grayscale images; RGB note below.
total_images = 0

for ch in channels:
    in_dir = img_root / ch
    out_dir = masks_root / ch / "png"
    ensure_dir(out_dir)

    if not in_dir.exists():
        print(f"[SKIP] Missing channel folder: {in_dir}")
        continue

    files = discover_images(in_dir, image_exts)
    if debug_max_images is not None:
        files = files[:debug_max_images]

    if not files:
        print(f"[WARN] No images in {in_dir} with {image_exts}")
        continue

    print(f"[{ch}] {len(files)} images -> {out_dir}")

    for group in tqdm(list(chunked(files, batch_size)), desc=f"Seg {ch}"):
        imgs = [io.imread(p) for p in group]

        # Cellpose 4.x accepts grayscale without 'channels'; let it infer.
        # Explicit 'channels' is deprecated warning; safest is omit it.
        masks, flows, styles = model.eval(
            imgs,
            diameter=diameter,                 # None = auto; avoids /0
            batch_size=len(imgs),
            channel_axis=None,                 # 2D arrays -> no channel axis
            flow_threshold=flow_threshold,
            cellprob_threshold=cellprob_threshold,
            normalize={"tile_norm_blocksize": tile_norm_blocksize},
        )

        for p, m in zip(group, masks):
            out_path = out_dir / f"{p.stem}_masks.png"
            io.imsave(out_path, m.astype(np.uint16), check_contrast=False)

    total_images += len(files)

print(f"Done. Segmented {total_images} images total.")
print(f"Masks written under: {masks_root}/<channel>/png/")


[red] 10 images -> /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/red/png


Seg red: 100%|██████████| 3/3 [02:59<00:00, 59.86s/it]


[yellow] 10 images -> /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/yellow/png


Seg yellow: 100%|██████████| 3/3 [03:10<00:00, 63.52s/it]


[blue] 10 images -> /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/blue/png


Seg blue: 100%|██████████| 3/3 [03:05<00:00, 61.76s/it]


[green] 10 images -> /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/green/png


Seg green: 100%|██████████| 3/3 [03:01<00:00, 60.50s/it]

Done. Segmented 40 images total.
Masks written under: /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/<channel>/png/


In [6]:
# Cell 6 — Sanity counts
for ch in channels:
    d = masks_root / ch / "png"
    n = len(list(d.glob("*_masks.png"))) if d.exists() else 0
    print(f"{ch}: {n} mask files")


red: 10 mask files
yellow: 10 mask files
blue: 10 mask files
green: 10 mask files
